## Make a figure showing all GHG data 
#### Including a fit a curve with the NOAA method
#### Including a seasonal cycle


The NOAA method for curve fitting used is described here: 

The method is described: https://gml.noaa.gov/ccgg/mbl/crvfit/crvfit.html 

The code is available at: https://gml.noaa.gov/aftp/user/thoning/ccgcrv/

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot

import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)
    

from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
import process_data
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload


from input.read_wdc_data import AvailableData, create_data_reader
#save figures in...
dir_save = '../output/timeseries/'

### First, read in all the GHG data

In [ ]:
## Read all data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


#### Remove outlies (e.g. CO peak)

In [ ]:
sel_species = ds_all.species
unique_species = np.unique(sel_species)
species_sel = unique_species

In [ ]:
# Remove outliers that exceed 10*stdedeviation and 4* the zscore (see )
ds_all_rem_out = process_data.rem_out(ds_all, std_fac=10, z_threshold=4)
## plot removed data
var = "value"

for s in species_sel:

    f, axs = plt.subplots(2, 1, sharex=True)
    plt.suptitle("Removed outliers")
    ds_all.sel(dataset=s)[var].plot(ls="", marker="o", ax=axs[0])
    ds_all.sel(dataset=s)[var + "_unc"].plot(ls="", marker="o", ax=axs[1])
    # new data:
    ds_all_rem_out.sel(dataset=s)[var].plot(ls="", marker=".", ax=axs[0])
    ds_all_rem_out.sel(dataset=s)[var + "_unc"].plot(ls="", marker=".", ax=axs[1])
    plt.show()

#### Prepare the figure

In [ ]:
## Define the data periods I want to compare/plot ##! TP ADAPT!

t0 = ds_all.isel(time=0)
t1 = ds_all.isel(time=-1)
print(f"Total time period with data:: {t0.time.values} to {t1.time.values}")

compare_periods = {
    #'A': (t0.time.values, dt.datetime(2006,12,31)), #old and flask data
    #'B': (dt.datetime(2008,1,1), dt.datetime(2011,12,31)), #only flask data
    'C': (dt.datetime(2020,1,1), t1.time.values), # only new data
    #'D': (dt.datetime(2014,1,1), t1.time.values) # only ozone data
}

In [ ]:
# Define fit paramaters for the curve fitting
# Default values
fit_params_defaults = {'shortterm': 80, #Short term cutoff value in days for smoothing of data
                'longterm': 667, # smoothing in days. Default: 667
                'numpolyterms': 2, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}

fit_properties = {}
for dataset in ds_all.dataset:
    dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset_name] = fit_params_defaults.copy()
    
# Update fit_properties with non-default values if necessary:
# Remove sampleinterval for flask data:
for dataset in fit_properties:
    if '_flask' in dataset:
        fit_properties[dataset]['sampleinterval'] = 0 #if 0, determine from xp (time)

#fit_properties['CO']['numpolyterms'] = 3


In [ ]:
## Helper functions for the figure

import matplotlib.colors as mc
# adjust the lightness of a color
def adjust_lightness(color, amount=0.5):
    import colorsys
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2])

# function to plot each subplot
# time series plot
def plot_data(ds_temp, ds_temp_fit, label, ax=None, detrend=False, **kwargs):
    ''' 
    Plot the data and the fit
    detrend: if True, plot the detrended data
    '''
    if ax is None:
        ax = plt.gca()

    ## get initial figure properties
    initial_color = kwargs['color']
    initial_ls = kwargs['ls']
    
    kwargs['ls'] = '' #no line for measurements
    ds_temp.plot(
        ax=ax,
        alpha=0.7,
        label=label,
        #markeredgewidth=0,
        **kwargs
    )
    #plot fit
    kwargs['color'] = adjust_lightness(initial_color,amount=1.5) # adapt hue of initial color (lighter)
    kwargs['marker'] = '' #no marker for fit
    kwargs['ls'] = initial_ls
    ds_temp_fit["smoothed_vals"].plot(ax=ax, label="curve fit", **kwargs)
    if detrend:
        #plot detrended fit
        kwargs['color'] = adjust_lightness(initial_color,amount=0.8)  # adapt hue of initial color (darker)
        kwargs['ls'] = ':' #dotted line for detrended
        ax.plot(
            ds_temp_fit.time,
            ds_temp_fit["seasonal_detrend"] + ds_temp_fit["smoothed_vals"].mean(),
            label="detrended fit",
            **kwargs,
        )  ## Add mean value to detrended to obtain same magnitude


# seasonality plot
def plot_cycle(ds_temp, ds_temp_fit, s, freq="month", ax=None, with_trend = False, **kwargs):
    '''
    Plot seasonal cycle of the data
    with_trend:  if True, plot the seasonal cycle with trend (non-detrended)
    '''
    
    ## get a different hue of the color used
    # Decrease the hue value
    initial_color = kwargs['color']
    
    if ax is None:
        ax = plt.gca()
    ref = (
        ds_temp.mean()
    )  # use mean value from measurements to obtain positive/neg. seasonality (ds_temp-ref) or absolute values (ds_temp_fit +ref)
    if freq=='hour':
        kwargs['color'] = adjust_lightness(initial_color,amount=1.5)  # adapt hue of initial color (lighter)
        ds_temp.groupby(f"time.{freq}").mean().plot(
            ax=ax, label="detrended daily cycle", 
            **kwargs
        )
    else:
        # normal seasonal cycle
        if with_trend:
            kwargs['ls'] = ':' #dotted line for not detrended
            kwargs['color'] = adjust_lightness(initial_color,amount=0.8) # adapt hue of initial color (darker)
            #plot also the normal seasonal cycle (with trend)
            (ds_temp-ref).groupby(f"time.{freq}").mean().plot(ax=ax, label="normal", **kwargs)
        # detrended seasonal cycle
        kwargs['color'] = adjust_lightness(initial_color,amount=1.5)  # adapt hue of initial color (lighter)
        ds_temp_fit["seasonal_detrend"].groupby(f"time.{freq}").mean().plot(
            ax=ax, label="detrended seasonal cycle", 
            **kwargs
        )


In [ ]:
import run_curve_fit
import numpy as np
import itertools
import matplotlib.dates as mdates
                
%autoreload 2


# dataset to use:
# ds_data = ds_all #normal measurement data
ds_data = ds_all_rem_out  # use data with removed outliers!!
#ds_data = ds_all_rem_out.sel(dataset=['CO','CO_flask'])#for debugging

##---- Plotting definitions ----##
save_fig = True
# plot seasonal or daily cycles
plot_seas = True #if false, plot daily cycle instead of seasonal
seas_freq = 'dayofyear' # 'month' or 'dayofyear'
with_trend = False #if true, plot detrended data

default_colors = ['C0', 'C1', 'C2', 'C3']
color_iterator = itertools.cycle(default_colors)



##--- Start figure  ---##
# Make a mosaic grid, where each subplot is called either the species (e.g. CO2) or the species seasonal cycle (ew.g. CO2_seas)
my_species = [ 'CO2', 'CH4', 'CO', 'O3']
mosaic_grids =  [[m, m+'_seas']  for m in my_species] # put [] around to start new row
gs_kw = dict(width_ratios=[3, 1]) #column and width ratios
fig, axd = plt.subplot_mosaic(mosaic_grids,
                              gridspec_kw=gs_kw, figsize=(8, len(my_species) * 2),
                              layout="constrained")
                              
for period, (start_date, end_date) in compare_periods.items():
    print(f"Period {period}: {start_date} to {end_date}")
    # Filter the data for the current period
    period_data = ds_data.sel(time=slice(start_date, end_date))

    # Define figure properties for this period
    fig_properties = {
        'color': next(color_iterator),
        'marker': '.',
        'ls':'-'
    }

    for i,dataset in enumerate(period_data.dataset):
        # Get the data for the current dataset and period
        data = period_data.sel(dataset=dataset)["value"]
        species = period_data.sel(dataset=dataset).species.values
        #print(f"Dataset: {dataset.item()}, Species: {species}")

        if data.size > 0 and np.all(np.isnan(data))==False: #if we have data
        
            # Define fit properties for this dataset
            # Perform the curve fitting and obtain the interpolated time axis
            filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
                ds=data,
                dataset_str=dataset.item(),
                t1=start_date,
                t2=end_date,
                **fit_properties[dataset.item()]
            )
              
            # Plot the data and the fit
            # adapt figure properties for flask data
            if dataset.item() == 'CO_flask':
                # CO_flask is the only flask data that we have in parallel with continuous measurements
                # Therefore, we plot it with a different marker and linestyle
                fig_properties['marker'] = 'd'
                #fig_properties['color'] = 'lightgrey'
                fig_properties['zorder'] = 5
                fig_properties['ls'] = '--'
            elif '_flask' in dataset.item():
                # different marker for flask measurements
                fig_properties['marker'] = 'd'
            else:
                # normal properties
                fig_properties['marker'] = '.'
                fig_properties['ls'] = '-'

           
            ## Time series figure
            ax_ts = axd[species.item()]
            ## Seasonal cycle figure
            ax_seas = axd[species.item()+'_seas']


            legend_entry = f"curve fit ({fit_properties[dataset.item()]['shortterm']} days)"
            plot_data(data,                       
                      ds_interp,
                      legend_entry, 
                      ax_ts,
                      with_trend
                      ,**fig_properties)

            #plot seasonal cycle
            freq = seas_freq if plot_seas else 'hour' #for seasonal cycle, use 'dayofyear' (smoother line) or 'month', for daily cycle (if plot_seas=False) use 'hour'
            # no markers for seasonal cycle
            fig_properties['marker'] = '' #no marker for seasonal cycle
            plot_cycle(
                data,
                ds_interp,
                s,
                freq= freq,
                ax=ax_seas,
                with_trend=with_trend,
                **fig_properties
            )


            #---- figure properties ----#

            # axes properties time series
            ax_ts.set_ylabel(f"{species} ({period_data.sel(dataset=dataset).unit.values})")
            #ax_ts.set_xlim(np.array([t1, t2], dtype="datetime64"))
            ax_ts.spines["top"].set_visible(False)
            ax_ts.spines["right"].set_visible(False)
            ax_ts.set_title("")

            if species != 'O3':  # all except lowest axis
                plt.setp(
                    ax_ts.get_xticklabels(), visible=False
                )  # remove xticklabels except for lowest plot
                ax_ts.set_xlabel("")

            # axes properties seasonal cycle
            # ax_seas.set_yticklabels('') #this removes ylabels for both axes! Therefore better use:
            #ax_seas.tick_params(labelleft=False) #I need the labels if I use other units for the seasoal cycle!!
            ax_seas.set_title("")
            ax_seas.set_ylabel("")
            ax_seas.spines["top"].set_visible(False)
            ax_seas.spines["right"].set_visible(False)
            if plot_seas:
                if freq== 'month':
                    months = range(1, 13, 3)
                    ax_seas.set_xticks(
                        months,
                        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
                    )
                elif freq == 'dayofyear':
                    ax_seas.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
                    ax_seas.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

            #ax_seas.grid(color='lightgrey',zorder=0,axis='x')
            ax_seas.set_axisbelow(False)

            # different for lowest plot
            if species != 'O3':  # all except lowest axis
                plt.setp(
                    ax_seas.get_xticklabels(), visible=False
                )  # remove xticklabels except for lowest plot
                ax_seas.set_xlabel("")

## --- Final figure properties ---##
handles, labels = ax_ts.get_legend_handles_labels()
# manually adapt legend:
labels[0] = "1h data"
ax0 = list(axd.values())[0] #first subplot (upper left)
if with_trend:
    ax0.legend(handles, labels, loc="lower right")
fig.align_ylabels()


# place title on first axis
ax0.set_title("Mt. Kenya measurements and curve fit", loc="left")
ax1 = list(axd.values())[1] #second subplot (upper right)
if plot_seas:
    tit_right = "Seasonal cycle \n (detrended)"
else:
    tit_right = 'Diurnal cycle'
ax1.set_title(tit_right, loc="center")


# # plot in each subplot a vertical line for the compare_periods (not working yet)
# for ax in axd.values():
#     if '_seas' in ax.get_label():
#         continue
#     else: 
#         for period, (start_date, end_date) in compare_periods.items():
#             ax.axvline(start_date, color='k', linestyle='--', alpha=0.5)
#             ax.axvline(end_date, color='k', linestyle='--', alpha=0.5)
#             # Create a rectangle patch with transparent background
#             #rect = plt.Rectangle((start_date, ax.get_ylim()[0]), end_date - start_date, ax.get_ylim()[1], facecolor=next(color_iterator), edgecolor='none')
#             # Add the rectangle patch to the plot
#             #ax.add_patch(rect)



str_seas = 'seas' if plot_seas else 'daily'
str_trend = '_detrended' if with_trend else ''
str_freq = '_' + seas_freq if plot_seas else ''
if save_fig:
    plt.savefig(
        f"{dir_save}timeseries_and_{str_seas}{str_freq}_fit{str_trend}_{len(compare_periods)}periods.png", dpi=300
    )